<a href="https://colab.research.google.com/github/miqqidami/testrepo/blob/main/CENG384_FFT_Ocean_Waves_Template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CENG384 Homework Template: Generating Ocean Waves with FFT

**Why ocean waves?**

Modern computer graphics often uses spectral methods and the Fast Fourier Transform to generate realistic open-ocean surfaces. Related wave-based and spectral techniques have influenced water effects in visual effects and games, including work associated with films such as *Titanic*, *Waterworld*, and *Life of Pi*. A classical reference is Jerry Tessendorf's  [*Simulating Ocean Water*](https://www.researchgate.net/publication/264839743_Simulating_Ocean_Water) (SIGGRAPH course notes), which describes how an ocean height field can be synthesized by defining wave amplitudes in the frequency domain and evaluating the resulting surface efficiently with FFTs.

**What you will build**

You will implement the building blocks of that pipeline from scratch — the 1D and 2D DFT/FFT, the shift property, frequency-domain filtering, and proper handling of sampling. By the end, you will have a small ocean-simulation demo that can generate a height field, drift it in a chosen direction, and filter it between calmer and choppier regimes.

If this assignment sparks your curiosity, Tessendorf's notes connect these computational steps to real ocean-simulation ideas such as spectral wave models, the Phillips spectrum, dispersion relations, and choppy waves via horizontal displacement. Many modern spectral ocean systems, including tools such as NVIDIA WaveWorks, follow the same broad frequency-domain-to-spatial-domain idea, although production water systems often combine FFTs with Gerstner waves, particle simulations, fluid solvers, foam models, and artist-directed controls.

In this programming assignment you have the following tasks:
* Task 0: Warmup (10 Points)
* Task 1: Write your own DFT/IDFT functions (15 Points)
* Task 2: Why we need the FFT (5 Points)
* Task 3: Implement Radix-2 FFT/IFFT (15 Points)
* Task 4: Extend your FFT to 2D (10 Points)
* Task 5: The shift property (15 Points)
* Task 6: Filtering (15 Points)
* Task 7: Sampling and Aliasing (15 Points)

**Getting Ready**
* Throughout the homework we will be using numpy library (check <a href="https://numpy.org/doc/2.0/user/quickstart.html">this</a> for a tutorial). Numpy is a very talented library for working with numerical data structures, like complex numbers, vectors, matrices etc.

* For plotting the signal, we will use the matplotlib library (check <a href="https://matplotlib.org/stable/users/explain/quick_start.html#a-simple-example">this</a> for a tutorial).

## Submission

Submit this completed notebook as `.ipynb`. Make sure the notebook includes **outputs**, **plots**, and **written explanations**.

Use the following filename format for your submission:
**{*surname*}\_{*name*}\_{*7_digit_studentid*}.ipynb**

## Plagiarism

Collaboration with other students and using generative AI tools are **forbidden**.

## Discussion

For your questions you can use the Discussion forum in Odtuclass.



Good luck,

Şeref Barış Çağlıyan,
Güneş Sucu,
Defne Odabaşı

## Setup

Some functions are provided for plotting, testing, and generating example surfaces. Do not modify them. Run the setup cell below first.

In [ ]:
# ── Setup: download course helpers (run this cell first) ──
!curl -sL -o ceng384_helpers.py https://raw.githubusercontent.com/barisc-ceng/CENG384_PA_2025_2/main/ceng384_helpers.py
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import importlib, ceng384_helpers
importlib.reload(ceng384_helpers)
from ceng384_helpers import *

## **Task 0: Warmup (10 Points)**

Lets generate and plot some simple signals to get ourselves familiar.

### **0.1 Creating a Continuous-Time Signal**

Your first task is to construct and plot the following signal:

$$x(t) = \cos(2\pi t) + 2\sin(3\pi t)$$

**TODO — complete the code cell below:**

1. **Sample** the time interval $[-2, 2)$ with a step size of $0.01$ using `np.arange`.
2. **Compute** $x(t)$ using `np.cos` and `np.sin`.
3. **Plot** $x(t)$ vs. $t$ as a line plot (`plt.plot`). Your figure must include:
   - x-axis label: `'Time (t)'`
   - y-axis label: `'Amplitude'`
   - A descriptive title (e.g. the equation)
   - A grid (`plt.grid`)

In [ ]:
# TODO 1: Sample the interval
t = None

# TODO 2: Compute x(t)
x = None

# TODO 3: Create a line plot of x vs t.
#         Add an x-label, y-label, title, and grid.

### **0.2 Creating Discrete-Time Signals**

A **discrete-time signal** is a sequence indexed by integers $n = 0, 1, \ldots, N-1$.
Unlike a continuous-time signal, there are no values *between* samples only the
sample points exist.

**TODO — implement `make_signal(signal_id, N)`** which generates a discrete-time
signal of length $N$. The function must return a 1-D NumPy **float** array:

| `signal_id` |&nbsp;&nbsp;&nbsp;&nbsp; formula &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;----- | description |
|---| :---|:---|
| 0 | $x_0[n] = \sin\!\bigl(\tfrac{2\pi \cdot 4\,n}{N}\bigr)$ | single sinusoid |
| 1 | $x_1[n] = \sin\!\bigl(\tfrac{2\pi \cdot 3\,n}{N}\bigr) + 0.4\,\cos\!\bigl(\tfrac{2\pi \cdot 12\,n}{N}\bigr)$ | two-component mix |
| 2 | $x_2[n] = \begin{cases}1, & N/4 \le n < N/2,\\ 0, & \text{otherwise}\end{cases}$ | rectangular pulse |

**Do not change the function signature or return type.**

---

After implementing `make_signal`, **visualize all three signals** in the plotting cell below.
Because these are discrete-time signals, use `plt.stem()` instead of `plt.plot()` —
stem plots display individual sample values rather than connecting them with a line.

> **Hint:** `plt.stem(n, x)` takes a sample-index array `n` and a values array `x`.
> Use `plt.subplots(3, 1)` to stack the three subplots vertically. Again do not forget to add axis labels, title for each subplot.


In [ ]:
def make_signal(signal_id, N):
    """
    Generate a discrete-time test signal of length N.

    Input:
      signal_id (int) : 0, 1, or 2 (see task description above)
      N (int) : number of samples

    Output:
      x : np.ndarray, shape (N,), dtype float64
    """
    n = np.arange(N)   # discrete indices 0, 1, ..., N-1

    if signal_id == 0:
        # TODO: single sinusoid
        x = None

    elif signal_id == 1:
        # TODO: sum of two sinusoids
        x = None

    elif signal_id == 2:
        # TODO: rectangular pulse
        x = None

    else:
        raise ValueError(f"Unknown signal_id {signal_id!r}. Choose 0, 1, or 2.")

    return x


In [ ]:
N_demo = 64
n_axis = np.arange(N_demo)  # sample indices 0, 1, ..., 63

# TODO: Plot all three signals (signal_id = 0, 1, 2) in three stacked subplots.
# Use plt.stem() for discrete-time signals not plt.plot()!
# Each subplot should have a title, axis labels, and a grid.

## **Task 1: Write your own DFT/IDFT functions (15 Points)**

Implement the $O(N^2)$ DFT and IDFT formulas. This is slower than FFT, but it helps you check the sign convention and normalization before writing the fast version.

### **1.1 Implementing DFT and IDFT from scratch**

You are expected to complete the given ```my_dft()``` and ```my_idft()``` functions.

The **analysis equation** (forward DFT) decomposes a length-$N$ signal into $N$ complex sinusoids:

$$
X[k] \;=\; \sum_{n=0}^{N-1} x[n]\, e^{-j\,2\pi k n / N}, \qquad k = 0, 1, \ldots, N-1.
$$

The **synthesis equation** (inverse DFT) rebuilds the signal from those coefficients:

$$
x[n] \;=\; \frac{1}{N}\sum_{k=0}^{N-1} X[k]\, e^{+j\,2\pi k n / N}, \qquad n = 0, 1, \ldots, N-1.
$$

**No Built-in Solvers: The use of ```np.fft.fft```, ```np.fft.ifft```, or any other high-level library functions that perform the Fourier Transform is strictly prohibited for this section.**

**Do not change functions arguments**

In [ ]:
def my_dft(x):
    """
    DFT, O(N^2).
    Input:
        x: 1D array-like signal
    Output:
        X_k: 1D complex-valued DFT coefficients

    Do NOT call np.fft.fft inside this function.
    """
    x = np.asarray(x, dtype=complex)
    N = x.size

    # TODO: implement the analysis equation
    raise NotImplementedError("Implement my_dft")


def my_idft(X_k):
    """
    Inverse DFT, O(N^2).
    Input:
        X_k: 1D array-like DFT coefficients
    Output:
        x: 1D complex-valued reconstructed signal

    Do NOT call np.fft.fft inside this function.
    """
    X_k = np.asarray(X_k, dtype=complex)
    N = X_k.size

    # TODO: implement the synthesis equation
    raise NotImplementedError("Implement my_idft")

### **1.2 Validate your Implementation**

Run the following cell to check if your implementations are correct.

**Do not change/add any code here**

In [ ]:
validate_task1(my_dft, my_idft)

### **1.3 Visualize your Implementation**
In this section, you will visualize a discrete-time signal’s journey from the **Time Domain** to the **Frequency Domain** and back to the **Time Domain** using your custom ```my_dft()``` and ```my_idft()``` functions.

We will use a composite signal consisting of three distinct frequencies and a DC offset:

$$x[n] = 3\sin(\frac{6\pi n}{N}) + 1.5\cos(\frac{14\pi n}{N}) + 0.8\sin(\frac{44\pi n}{N}) + 2.$$


**Tasks:**
1. Sampling: Generate the signal for each sampling frequency \(f_s in \{15, 50, 100\}\) Hz over a duration of 1 second.
    * *Note*: Since the duration is 1 second, use \(N=f_s\) samples and define the sample indices as \(n = 0, 1, ..., N-1\).
2. Transform: Compute the DFT of the sampled signal using your `my_dft()` function.
3. Inverse Transform: Reconstruct the signal using your `my_idft()` function.
4. Visualization: For each value of \(f_s\), create a figure with two subplots:

    * Subplot 1: The magnitude of the DFT coefficients, \(|X[k]|\), versus the frequency-bin index \(k = 0, 1, ..., N-1\).
    * Subplot 2: The reconstructed signal from IDFT versus the sample index \(n = 0, 1, ..., N-1\).

In [ ]:
# Reference signal at fs=100 — run this to see the target waveform
fs = 100
n = np.arange(fs)
x_n = (2.0
       + 3.0 * np.sin(2 * np.pi * 3  * n / fs)
       + 1.5 * np.cos(2 * np.pi * 7  * n / fs)
       + 0.8 * np.sin(2 * np.pi * 22 * n / fs))

plt.figure(figsize=(10, 3))
plt.stem(n, x_n, basefmt="k-")
plt.title(f"Reference signal x[n]  (N={fs} samples)")
plt.xlabel("Sample index n"); plt.ylabel("Amplitude")
plt.grid(True, alpha=0.3); plt.show()

# Your implementation below
fs_list = [15, 50, 100]

for fs in fs_list:
    n = np.arange(fs)

    # TODO 1: Define x_n using the same formula as above, sampled at this fs

    # TODO 2: Compute the DFT with my_dft

    # TODO 3: Reconstruct the signal with my_idft

    # TODO 4: Create a figure with 2 subplots
    #   Subplot 1 — DFT magnitude |X[k]| vs frequency bin k
    #   Subplot 2 — Reconstructed signal x_recon[n] vs n
    #   Each subplot needs a title, axis labels, and a grid.


### **1.4 Discussion**

**Q1a.** What is the minimum sampling frequency (fs​) required to perfectly reconstruct this specific signal without aliasing? Explain your reasoning.

**Q1b.** The DFT has a large spike at $0 Hz$. How does its magnitude relate to the constant term in your signal equation and the number of samples?

**Q1c.** The DFT magnitude plot is symmetric around `fs/2`. Explain the mathematical property of real-valued signals that causes this mirror effect.

## Answers

**A1a.**

**A1b.**

**A1c.**

## **Task 2: Why we need Fast-Fourier Transform (FFT)? (5 Points)**

The analysis equation evaluates $N$ output bins, each requiring a sum of $N$ terms, so the DFT costs

$$T_{\text{DFT}}(N) = \Theta(N^{2}).$$

Doubling $N$ should make the runtime approximately $4\times$ larger.


Run the timing cell below, then complete the discussion.

**Do not change/add any code here**

In [ ]:
run_timing_comparison(my_dft, make_signal)

### **2.1 Discussion**

**Q2a.** Look at the log-log plot. When the input size N increased by one decade (e.g. from 100 to 1000), how many decades does the execution time for `my_dft` increase? Explain why this specific relationship (the slope) is consistent with $\Theta(N^{2})$?

**Q2b.** Imagine you are processing a digital image of size $N\times N$. To compute its 2D-DFT, you must perform a 1D transform on every row, then every column. How many 1D transforms does that require? Write the total cost of the 2D **DFT** in $\Theta$-notation?

## Answers

**A2a.**

**A2b.**

## **Task 3: Implement Radix-2 FFT and IFFT (15 Points)**

As you observed in Task 2, the DFT costs $\Theta(N^2)$. The **Radix-2 Fast Fourier Transform** (Cooley–Tukey algorithm) reduces this to $\Theta(N \log N)$ by recursively exploiting the structure of the DFT.

> **Requirement:** the input length $N$ must be a power of 2 (e.g., 2, 4, 8, 16).

---

### **3.1 Derivation**

**Step 1 — Start from the DFT definition**

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-j2\pi kn/N}, \qquad k = 0, 1, \ldots, N-1.$$

---

**Step 2 — Split the sum into even-indexed and odd-indexed samples**

Separate the sum over $n$ into even indices $n = 2m$ and odd indices $n = 2m+1$, each running from $m = 0$ to $N/2 - 1$:

$$X[k] = \sum_{m=0}^{N/2-1} x[2m]\, e^{-j2\pi k(2m)/N}\; +\; \sum_{m=0}^{N/2-1} x[2m+1]\, e^{-j2\pi k(2m+1)/N}.$$

---

**Step 3 — Factor the common multiplier factor from the odd sum**

The odd sum contains a factor $e^{-j2\pi k/N}$ that does not depend on $m$. Pull it out:

$$X[k] = \sum_{m=0}^{N/2-1} x[2m]\, e^{-j2\pi km/(N/2)}\; +\; e^{-j2\pi k/N} \sum_{m=0}^{N/2-1} x[2m+1]\, e^{-j2\pi km/(N/2)}.$$

Notice that $e^{-j2\pi k(2m)/N} = e^{-j2\pi km/(N/2)}$ — leaving a DFT of half the length.

---

**Step 4 — Recognise the two half-length DFTs**

Define:
- $E[k]$ = DFT of the **even-indexed** subsequence $\bigl(x[0],\, x[2],\, x[4],\, \ldots,\, x[N-2]\bigr)$, length $N/2$
- $O[k]$ = DFT of the **odd-indexed** subsequence $\bigl(x[1],\, x[3],\, x[5],\, \ldots,\, x[N-1]\bigr)$, length $N/2$

Then:

$$X[k] = E[k] + e^{-j2\pi k/N}\cdot O[k], \qquad k = 0, 1, \ldots, N-1.$$

---

**Step 5 — Use periodicity and the anti-symmetry of the twiddle factor**

$E[k]$ and $O[k]$ are each periodic with period $N/2$:

$$E[k + N/2] = E[k], \qquad O[k + N/2] = O[k].$$

The twiddle factor satisfies:

$$e^{-j2\pi(k+N/2)/N} = e^{-j2\pi k/N}\cdot e^{-j\pi} = -\,e^{-j2\pi k/N}.$$

---

**Step 6 — The butterfly equations**

Combining Steps 4 and 5, all $N$ output bins follow from just $N/2$ multiplications.
For $k = 0, 1, \ldots, N/2 - 1$:

$$\boxed{X[k]\phantom{{}+N/2} = E[k] + e^{-j2\pi k/N} \cdot O[k]}$$

$$\boxed{X[k + N/2] = E[k] - e^{-j2\pi k/N} \cdot O[k]}$$

This pair of equations is called a **butterfly**.

---

**Step 7 — Recursion gives $O(N \log N)$**

Apply the same even/odd split recursively to $E[k]$ and $O[k]$, until reaching length-1 DFTs (which are trivial: $X[0] = x[0]$). If $N = 2^p$, there are $p = \log_2 N$ levels, each doing $O(N)$ work:

$$T_{\text{FFT}}(N) = O(N \log_2 N).$$

---

### **3.2 Inverse FFT**

The inverse DFT synthesis equation is:

$$x[n] = \frac{1}{N}\sum_{k=0}^{N-1} X[k]\, e^{+j2\pi kn/N}.$$

**Hint**: A clean implementation trick uses the **conjugation identity**.

---

### **3.3 Your Task**

Implement `my_fft(x)` and `my_ifft(X)` recursively in the cell below.

**Do not call `np.fft.fft` or `np.fft.ifft` inside your functions.**

In [ ]:
def my_fft(x):
    """
    Radix-2 Cooley-Tukey FFT (recursive).
    Input :
      x: 1-D array, length must be a power of 2
    Output:
      X: 1-D complex FFT coefficients

    Do NOT call np.fft.fft inside this function.
    """
    x = np.asarray(x, dtype=complex)
    N = x.size

    # TODO: implement the radix-2 FFT
    raise NotImplementedError("Implement my_fft")


def my_ifft(X):
    """
    Inverse FFT
    Do NOT call np.fft.ifft inside this function.
    """
    X = np.asarray(X, dtype=complex)
    N = X.size

    # TODO: apply the conjugation identity
    raise NotImplementedError("Implement my_ifft")


### **3.4 Validate your Implementation**

Run the following cell to check if your implementations are correct.

**Do not change/add any code here**

In [ ]:
validate_task3(my_fft, my_ifft)

## **Task 4: Extend your FFT to 2D (10 Points)**

### **4.1 The 2D DFT and Separability**

For an $N \times N$ array $x[n_1, n_2]$, the 2D DFT is defined as:

$$X[k_1, k_2] = \sum_{n_1=0}^{N-1}\sum_{n_2=0}^{N-1} x[n_1, n_2]\;e^{-j2\pi k_1 n_1/N}\;e^{-j2\pi k_2 n_2/N}.$$

Because the exponent separates into an independent term per axis, the inner sum over $n_2$ is simply a **1D DFT along each row**, and the outer sum over $n_1$ is a **1D DFT along each column** of the result:

$$X[k_1, k_2] = \sum_{n_1=0}^{N-1} e^{-j2\pi k_1 n_1/N} \underbrace{\left[\sum_{n_2=0}^{N-1} x[n_1, n_2]\,e^{-j2\pi k_2 n_2/N}\right]}_{\text{1D DFT of row }}.$$

This means you do **not** need a new algorithm — you apply your `my_fft` twice:

1. Apply `my_fft` to every **row** → intermediate result $R$
2. Apply `my_fft` to every **column** of $R$ → final spectrum $X$

The same logic applies to `my_ifft2` using `my_ifft`.

### **4.2 Your Task**

Complete `my_fft2` and `my_ifft2` in the cell below using only your `my_fft` and `my_ifft`.

Assume that both dimensions of the input array (N x N) must be powers of 2.

**Do not call `np.fft.fft2`, `np.fft.fft`, or any other library FFT.**


In [ ]:
def my_fft2(x2):
    """
    2D FFT via row-then-column 1D FFTs.
    Input :
      x2: 2D array, both dimensions must be powers of 2
    Output:
      X2: 2D complex spectrum (same shape as 'x2')

    Do NOT call np.fft.fft2 or np.fft.fft inside this function.
    """
    x2 = np.asarray(x2, dtype=complex)
    N0, N1 = x2.shape

    raise NotImplementedError("Implement my_fft2")


def my_ifft2(X2):
    """
    2D inverse FFT via row-then-column 1D IFFTs.
    Input :
      X2: 2D complex spectrum
    Output:
      x2: 2D complex reconstructed spatial signal

    Do NOT call np.fft.ifft2 or np.fft.ifft inside this function.
    """
    X2 = np.asarray(X2, dtype=complex)
    N0, N1 = X2.shape

    raise NotImplementedError("Implement my_ifft2")

### **4.3 Validate your Implementation**

Run the following cell to check if your implementations are correct.

**Do not change/add any code here**

In [ ]:
validate_task4(my_fft2, my_ifft2)

### **4.4 Visualize your Implementation**

Run the following cell to visualize your implementation.

**Do not change/add any code here**

In [ ]:
visualize_task4(my_fft2, my_ifft2)

### **4.5 Discussion**

**Q4a.** Why are the spectral peaks now distributed along both the horizontal and vertical axes instead of just one?

## Answers

**A4a.**

## **Task 5: The shift property: making the waves drift (15 Points)**

### **5.1 2D space-shift property**

The **2D space-shift property** of the DFT says that shifting a signal spatially by $(\Delta x, \Delta y)$ is equivalent to multiplying its spectrum by a linear-phase term:

$$
z[m-\Delta y,\;n-\Delta x]
\;\stackrel{\text{DFT2}}{\longleftrightarrow}\;
H[k_y, k_x]\;\cdot\;(\text{phase factor involving }k_x,\Delta x,k_y,\Delta y).
$$

Recall that for a domain of physical length $L$, the continuous wavenumber for DFT bin $p$ is $k = 2\pi p / L$.

We use this to translate the surface in time: at time $t$, the waves have drifted by

$$
\Delta x(t) = c\,t\cos\theta, \qquad \Delta y(t) = c\,t\sin\theta,
$$

where $c$ is the wave speed and $\theta$ is the propagation direction.

### **5.2 Your Task**

1. `translate_spectrum(H, L, dx, dy)` returns the spectrum of the spatially shifted signal given its current spectrum `H` on an $L \times L$ domain and shifts `dx`, `dy`. A helper that converts DFT bin indices to physical wavenumbers is already provided in the notebook.
2. `moving_surface(Z, L, t, theta_deg, speed)` returns the real-valued ocean surface that has drifted for time `t`. Use `my_fft2` / `my_ifft2` from Part 4 and `translate_spectrum` above.

In [ ]:
def translate_spectrum(H, L=10.0, dx=0.0, dy=0.0):
    H = np.asarray(H, dtype=complex)
    N = H.shape[0]
    KX, KY = make_wavenumber_grid(N, L=L)
    # TODO
    raise NotImplementedError("Implement translate_spectrum")

def moving_surface(Z, L=10.0, t=0.0, theta_deg=0.0, speed=0.5):
    # TODO
    raise NotImplementedError("Implement moving_surface")

### **5.3 Visualize your Implementation**

Run the following cell to visualize your implementation.

**Do not change/add any code here**

In [ ]:
animate_task5(moving_surface)

### **5.4 Discussion**

**Q5a.** After calling `translate_spectrum`, does the magnitude spectrum `|H[ky,kx]|` change? Explain why or why not, using the shift property formula.


**Q5b.** Suppose you call `translate_spectrum` twice in a row, first with $(\Delta x_1, \Delta y_1)$ and then with $(\Delta x_2, \Delta y_2)$. What single call would produce the same result? What general Fourier property does this illustrate?

## Answers

**A5a.**


**A5b.**

## **Task 6: Filtering (15 Points)**

### **6.1 Filtering**

The **2D convolution theorem** states that applying a linear shift-invariant filter in the spatial domain is equivalent to multiplying by the filter's 2D DFT in the frequency domain:

$$
Z_{\text{filt}}[m,n] \;=\; (z * h)[m,n]
\;\Longleftrightarrow\;
\widehat Z_{\text{filt}}[k_y,k_x] \;=\; \widehat Z[k_y,k_x] \cdot H_f[k_y,k_x].
$$

We use ideal binary masks based on the **radial DFT-bin distance**

$$R[k_x,k_y] = \sqrt{k_x^{2}+k_y^{2}},$$

where $k_x, k_y$ are **signed** bin indices following the standard DFT output order. Three mask types:

$$
M_{\text{low}} = \mathbf{1}\{R \le c\}, \quad
M_{\text{high}} = \mathbf{1}\{R \ge c\}, \quad
M_{\text{band}} = \mathbf{1}\{c_\ell \le R \le c_h\}.
$$

### **6.2 Your Task**

1. Implement `radial_bin_grid(N)` returning the $N\times N$ array of radial bin distances $R[k_x,k_y]$. Note that bin indices are **signed** integers (both positive and negative frequencies appear in the DFT output).
2. Implement `make_frequency_mask(N, mode, cutoff, low, high)` returning a float $\{0,1\}$ array. 'mode' is one of the three mask types {low, high, band}.
3. Implement `filter_surface(Z, mode, cutoff, low, high)` returning `(Z_filtered, mask)` using `my_fft2` and `my_ifft2` only.

In [ ]:
def radial_bin_grid(N):
    bins = frequency_indices(N)
    # TODO
    raise NotImplementedError("Implement radial_bin_grid")

def make_frequency_mask(N, mode="low", cutoff=8, low=4, high=12):
    # TODO
    raise NotImplementedError("Implement make_frequency_mask")

def filter_surface(Z, mode="low", cutoff=8, low=4, high=12):
    # TODO
    raise NotImplementedError("Implement filter_surface")

### **6.3 Visualize your Implementation**

Run the following cell to visualize your implementation.

**Do not change/add any code here**

In [ ]:
visualize_task6(my_fft2, filter_surface)

### **6.4 Discussion**

**Q6a.** The mask is defined using the *radial* bin distance $R = \sqrt{k_x^2 + k_y^2}$, producing a circular boundary in the frequency plane. What kind of spatial patterns does a circular low-pass mask preserve? How would the filtered result differ if you used a *rectangular* mask (separate independent cutoffs on $k_x$ and $k_y$)?

**Q6b.** After the low-pass filter is applied, which spatial scales survive and which are removed? Relate the cutoff bin number to the physical wavelength on the $L\times L$ domain.

## Answers

**A6a.**

**A6b.**

## **Task 7: Sampling and aliasing (15 Points)**

### **7.1 Sampling**

Decimating by integer factor $D$ — keeping every $D$-th row and column —

$$Z_{\downarrow D}[m,n] = Z[Dm,\,Dn]$$

reduces the number of samples from $N\times N$ to $(N/D)\times(N/D)$. This lowers the effective spatial sampling rate and changes the range of frequencies that can be represented without ambiguity.

Any frequency content in the original signal that exceeds the new representable limit **folds back (aliases)** into a lower apparent frequency in the decimated signal. The standard remedy is to remove those frequencies **before** decimating.

### **7.2 Your Task**

1. Implement `downsample2d(Z, factor)` returning the decimated array.
2. Implement `add_ripple(Z, amplitude, fx, fy)` adding the high-frequency component $a\cos\!\bigl(2\pi(f_x n + f_y m)/N\bigr)$ to the surface.
3. Implement `antialias_downsample(Z, factor)` determining the appropriate frequency limit for the given decimation factor, apply a low-pass filter using `filter_surface` from Part 6, then decimate.
4. Run the visualisation cell, then answer the reflection questions below.

In [ ]:
def downsample2d(Z, factor=2):
    # TODO
    raise NotImplementedError("Implement downsample2d")

def add_ripple(Z, amplitude=0.06, fx=19, fy=15):
    # TODO
    raise NotImplementedError("Implement add_ripple")

def antialias_downsample(Z, factor=2):
    # TODO
    raise NotImplementedError("Implement antialias_downsample")


### **7.3 Visualize your Implementation**

Run the following cell to visualize your implementation.

**Do not change/add any code here**

In [ ]:
visualize_task7(add_ripple, downsample2d, antialias_downsample, filter_surface)

### **7.4 Discussion**

**Q7a.** Describe the visual difference between the directly decimated surface and the anti-aliased surface. What artifact is present in the direct decimation result, and where does it come from?

**Q7c.** The anti-aliasing step is implemented entirely in the frequency domain (mask multiply + IDFT). Estimate its computational cost in $\Theta$-notation and compare it to an equivalent spatial-domain convolution. Why is the frequency-domain approach preferred?


## Answers

**A7a.**

**A7b.**
